# Prioritized Experience Replay (PER) - Empirische Analyse

## CAS AI Workshop - DQN Extensions

Dieses Notebook dokumentiert eine umfassende empirische Untersuchung von Prioritized Experience Replay (PER) im Vergleich zu Uniform Replay.

## 1. Konzeptionelle Erklärung: Was ist PER?

### Motivation

Nicht alle Erfahrungen im Replay Buffer sind gleich wertvoll:
- **Leichte Samples**: Agent versteht bereits gut → wenig Lerngain
- **Schwierige Samples**: Agent ist überrascht → großes Lerngain

**Uniform Replay** sampelt alle Experiences mit gleicher Wahrscheinlichkeit.
**Prioritized Replay** sampelt wichtige Experiences häufiger.

### Prioritätsdefinition

Priorität basiert auf **TD-Error** (Temporal Difference Error):

$$p_i = (|TD_i| + \epsilon)^\alpha$$

wobei:
- $TD_i = r_i + \gamma \max_{a'} Q(s', a') - Q(s, a)$ (Überraschung)
- $\alpha \in [0, 1]$ kontrolliert Prioritäts-Strength
- $\epsilon$ verhindert Zero-Priority für erneute Samples

## 2. Replay Buffer Mechanik

### Uniform Replay Buffer

```python
class UniformReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)  # FIFO
    
    def add(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)  # Uniform
        return batch
```

**Eigenschaften:**
- O(1) Insert
- O(1) Sample (mit Indexing)
- Speicher: O(N)
- Keine Prioritäts-Updates

### Prioritized Replay Buffer (mit SumTree)

```python
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha):
        self.tree = SumTree(capacity)  # Binärbaum
        self.alpha = alpha
    
    def add(self, s, a, r, s2, done, priority):
        self.tree.add(priority^alpha, data)  # Log-Update
    
    def sample(self, batch_size, beta):
        # Stratified sampling mit Importance Weights
        batch = stratified_sample(self.tree, batch_size)
        weights = (1 / (N * P(i)))^beta  # IS-Korrektur
        return batch, weights
```

**Eigenschaften:**
- O(log N) Insert
- O(log N) Sample
- O(log N) Priority Update
- Speicher: O(N) für Tree + O(N) für Data

## 3. Importance Sampling Correction

### Problem mit Prioritized Sampling

Wenn wir Samples mit höherer Wahrscheinlichkeit sampeln, ändern wir die Verteilung:
- **Uniform Sampling**: P(i) = 1/N für alle i
- **Prioritized Sampling**: P(i) ∝ p_i

Dies biased die Gradienten-Schätzung! Lösung: Importance-Sampling Weights

### Gewichtungsformel

$$w_i = \left( \frac{1}{N \cdot P(i)} \right)^\beta$$

wobei:
- $\beta \in [0, 1]$ interpoliert zwischen uniform (β=0) und unbiased (β=1)
- $N$ = Buffer Size
- $P(i)$ = Sampling-Wahrscheinlichkeit

### Praktische Implementierung

```python
# Im Training
probs = priorities / sum(priorities)
weights = (1 / (N * probs)) ** beta
weights = weights / max(weights)  # Normalize

# Loss mit Weights
loss = mean(weights * td_error^2)
```

## 4. Experimentelles Setup

### Environment
- **LunarLander-v3** (Gymnasium)
- State: [x, y, vx, vy, angle, angular_velocity, left_leg, right_leg] (8 dims)
- Actions: 0=Idle, 1=Left Engine, 2=Right Engine, 3=Main Engine (4 actions)
- Reward: Negative distance to goal, fuel cost, crash penalty
- Goal: Return > 200 (soft landing)

### Hyperparameter
```
TOTAL_STEPS = 300,000
BATCH_SIZE = 256
BUFFER_SIZE = 200,000
LEARNING_RATE = 2e-3
GAMMA = 0.99  # Discount factor
TAU = 0.005   # Soft update target network

EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 30,000 steps

# PER Specific
PER_ALPHA = 0.4  # Reduced for stability
PER_BETA_START = 0.4
PER_BETA_END = 1.0
PER_BETA_STEPS = 60,000
```

### Evaluation
- Alle 15,000 Steps
- 5 Evaluation Episodes (greedy, no exploration)
- Report: Mean ± Std Return

## 5. Trainingsergebnisse

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load results
with open('results_summary_CORRECTED.json') as f:
    results = json.load(f)

uniform = results['uniform']
per = results['per']

# Extract data
u_means = np.array(uniform['eval_means'])
u_stds = np.array(uniform['eval_stds'])
u_steps = np.array(uniform['eval_steps'])

p_means = np.array(per['eval_means'])
p_stds = np.array(per['eval_stds'])
p_steps = np.array(per['eval_steps'])

# Print summary
print('='*60)
print('ERGEBNISSE: Uniform Replay vs. PER')
print('='*60)
print(f'\nUniform Replay:')
print(f'  Final: {u_means[-1]:.2f} ± {u_stds[-1]:.2f}')
print(f'  Max:   {np.max(u_means):.2f}')
print(f'  Mean:  {np.mean(u_means):.2f} ± {np.std(u_means):.2f}')
print(f'\nPrioritized Experience Replay:')
print(f'  Final: {p_means[-1]:.2f} ± {p_stds[-1]:.2f}')
print(f'  Max:   {np.max(p_means):.2f}')
print(f'  Mean:  {np.mean(p_means):.2f} ± {np.std(p_means):.2f}')
print(f'\nDifferenz (PER - Uniform):')
print(f'  Final: {p_means[-1] - u_means[-1]:+.2f}')
print(f'  Percent: {((p_means[-1] - u_means[-1]) / abs(u_means[-1]) * 100):+.1f}%')

## 6. Visualisierungen

In [ ]:
# Main comparison with confidence bands
fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(u_steps, u_means, 'o-', color='#1f77b4', label='Uniform Replay', linewidth=3, markersize=8)
ax.fill_between(u_steps, u_means - u_stds, u_means + u_stds, color='#1f77b4', alpha=0.2)

ax.plot(p_steps, p_means, 's-', color='#ff7f0e', label='PER', linewidth=3, markersize=8)
ax.fill_between(p_steps, p_means - p_stds, p_means + p_stds, color='#ff7f0e', alpha=0.2)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Training Steps', fontsize=12, fontweight='bold')
ax.set_ylabel('Evaluation Return', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison: Uniform Replay vs. PER', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()

## 7. Beobachtungen & Überraschungen

### Überraschender Fund: Uniform Replay ist stabiler!

**Erwartung:** PER sollte besser sein (höhere Rewards)
**Beobachtung:** Uniform Replay zeigt konsistentere Performance

### Hypothesen für dieses Phänomen:

#### 1. **LunarLander ist relativ einfach**
- Uniform Sampling meist ausreichend
- PER vorteilhaft erst bei komplexeren Tasks (z.B. Atari)

#### 2. **PER ist hyperparameter-sensitiv**
- Alpha=0.4 könnte suboptimal sein
- Beta-Schedule könnte besser tuned werden
- Zu aggressive Priorisierung → Instabilität

#### 3. **Sampling Bias bleibt trotz Importance Weights**
- Häufig-gesampelte Transitions können overfit werden
- Importance-Sampling Correction ist nur asymptotisch unbiased

#### 4. **TD-Error basierte Prioritäten können oszillieren**
- Early in training: alle TD-Errors sind groß
- Später: TD-Error wird noisy wenn Agent am Limit ist
- Kann zu Oversampling von "edge cases" führen

### Beobachtete Muster:

1. **Beide Methoden lernen:** Rewards steigen über Zeit
2. **Uniform stabiler:** Niedrigere Varianz zwischen Evaluationen
3. **PER volatiler:** Höhere Spitzen und Täler
4. **Keine dominante Methode:** Abhängig von Punkt im Training

## 8. Statistische Analyse

In [ ]:
from scipy import stats

# T-Test
t_stat, p_value = stats.ttest_ind(p_means, u_means)

print('T-TEST: Sind die Mittelwerte signifikant unterschiedlich?')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value: {p_value:.6f}')
print(f'  Signifikant (α=0.05)? {"Ja" if p_value < 0.05 else "Nein"}')

# Effect size (Cohen's d)
cohens_d = (np.mean(p_means) - np.mean(u_means)) / np.sqrt((np.std(p_means)**2 + np.std(u_means)**2) / 2)
print(f'\nCohen\'s d (Effektgröße): {cohens_d:.3f}')
if abs(cohens_d) < 0.2:
    effect = 'Minimal'
elif abs(cohens_d) < 0.5:
    effect = 'Klein'
elif abs(cohens_d) < 0.8:
    effect = 'Mittel'
else:
    effect = 'Groß'
print(f'  Interpretation: {effect}')

# Variance comparison
u_var = np.var(u_means)
p_var = np.var(p_means)
f_stat = max(p_var, u_var) / min(p_var, u_var)
p_variance = 1 - stats.f.cdf(f_stat, len(p_means)-1, len(u_means)-1)

print(f'\nVARIANZ-TEST (Levene\'s Test):')
print(f'  Uniform Variance: {u_var:.2f}')
print(f'  PER Variance: {p_var:.2f}')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  Signifikant unterschiedlich? {"Ja" if p_variance < 0.05 else "Nein"}')

## 9. Reflexion: Wann hilft PER besonders?

### ✓ PER ist ideal für:

1. **Komplexe Umgebungen**
   - Atari-Spiele (57 actions, complex visual features)
   - Robotik mit vielen Freiheitsgraden
   - Multi-Agent Settings

2. **Sparse oder delayed Rewards**
   - Nur wenige erfolgreiche Trajectories
   - Große Reward-Gradienten zwischen ähnlichen States
   - Z.B. Monte Carlo Tree Search + RL

3. **Große Replay Buffer**
   - > 1 Millionen Transitions
   - Computational cost of O(log N) negligible

4. **Heterogene Schwierigkeit**
   - Sehr unterschiedliche TD-Errors in Buffer
   - Agent sieht mix of easy und hard scenarios

### ✗ PER kann problematisch sein in:

1. **Einfachen Umgebungen**
   - Kontinuierliche Kontrolle mit Richtungssignal
   - Diskrete Probleme mit klarem Gradient
   - Zu viel Komplexität für zu kleinen Benefit

2. **Stabilitäts-kritischen Anwendungen**
   - Roboter mit realen Kosten
   - Sicherheits-kritische Systeme
   - Uniform Replay zuverlässiger

3. **Mit unzureichendem Tuning**
   - Falsche Alpha/Beta → Instabilität
   - Fehlerhafte SumTree-Implementierung
   - Fehlende IS-Correction

## 10. Wann wird PER instabil?

### Instabilitäts-Szenarien:

#### 1. **Zu großes ALPHA (z.B. α ≥ 0.8)**
```
p_i = (|TD_i| + ε)^0.8
```
- Extrem starke Priorisierung
- Ein paar Samples dominieren Buffer
- Kann zu Mode Collapse führen
- → Gradient wird spiky, instabiles Training

#### 2. **Zu kleines BETA (z.B. β < 0.1)**
```
w_i = (1 / (N * P(i)))^0.05  # Fast keine Correction!
```
- Unzureichende Importance-Sampling Correction
- Bias in Gradient-Schätzung bleibt
- → Training divergiert von optimalen Policy

#### 3. **Falsche TD-Error Schätzung**
- Initial: Target-Netzwerk sehr schlecht
- → Alle TD-Errors sind riesig
- → SumTree sampling ist fast uniform anyway!
- → Oder: Zu aggressive Gradient-Updates

#### 4. **Kleine Batch Sizes (< 32)**
- Mit PER: noch höhere Varianz
- Wichtige Samples könnten übersampelt werden
- → Overfitting auf einzelne trajectories

#### 5. **Nicht-stationäre Umgebung**
- TD-Errors change dramatically
- Priorities werden schnell obsolet
- Gradienten können instabil werden

## 11. Nachteile von PER

### 1. **Computational Cost**
- SumTree: O(log N) vs. O(1) sampling
- Bei BUFFER_SIZE=1M: ~20 Operationen pro Sample
- Overhead: ~2-3x langsamer als Uniform
- Problem für Real-Time Applications

### 2. **Hyperparameter Komplexität**
- Uniform Replay: Einfach (nur Batch Size)
- PER: Alpha, Beta, Beta-Schedule, Epsilon
- Jeder Parameter ändert Behavior drastisch
- Hyperparameter-Suche wird 10x aufwendiger

### 3. **Memory Overhead**
- SumTree braucht 2N-1 Knoten
- Plus leaf indices für Updates
- ~2x Memory von Uniform Replay

### 4. **Implementierungs-Komplexität**
- SumTree Index-Management subtil
- Importance-Sampling Weights richtig implementieren
- Priority Updates korrekt zuordnen
- Viele Möglichkeiten für Bugs (wir hatten einen!)

### 5. **Bias durch Häufiges Sampling**
- Wichtige Samples werden häufig gezogen
- Können zu Overfitting auf diese führen
- IS-Weights sind nur asymptotisch unbiased
- Small sample size → bias bleibt

### 6. **Nicht für alle Problem-Typen ideal**
- Nicht universell besser als Uniform
- Stark Problem-abhängig
- Falsche Erwartung: "PER = immer besser"

## 12. Best Practices & Empfehlungen

### Entscheidungsbaum: Wann welche Methode?

```
Einfache Umgebung (< 50 States, Clear Reward Signal)?
  ├─ JA → Verwende UNIFORM REPLAY
  │        (Einfach, schnell, robust)
  │
  └─ NEIN
       │
       Sparse/Delayed Rewards?
       ├─ JA → Verwende PER
       │        (Fokus auf "surprising" events)
       │
       └─ NEIN
            │
            Buffer > 500K transitions?
            ├─ JA → Evtl. PER (O(log N) cost negligible)
            └─ NEIN → Uniform Replay suffiziert
```

### Hyperparameter Startpunkte:

**PER Baseline:**
```
PER_ALPHA = 0.4        # Moderate prioritization
PER_BETA_START = 0.4   # Start with 60% correction
PER_BETA_END = 1.0     # Full correction at end
PER_EPS = 1e-6         # Prevent zero-priority
```

**Wenn zu instabil:**
```
PER_ALPHA = 0.2        # Reduce to 0.2
  + Erhöhe GRAD_CLIP_NORM
  + Reduziere LEARNING_RATE
```

**Wenn zu langsam:**
```
PER_ALPHA = 0.6        # Increase to 0.6
  + Reduziere GRAD_CLIP_NORM (vorsichtig!)
```

### Monitoring-Metriken:

1. **Reward Curve**: Sollte monotoniesch steigen
2. **Loss**: Sollte abfallen, dann stabilisieren
3. **Gradient Norm**: Sollte nicht explodieren
4. **TD-Error Distribution**: Überprüfe auf Mode Collapse
5. **Sample Frequencies**: Überprüfe ob ein paar Samples überladen werden

## 13. Zusammenfassung

### Key Findings:

✓ **PER ist theoretisch elegant**
- Fokussiert Training auf schwierige Samples
- Gut motiviert aus Information-Theorie

✓ **Empirisch: Kontext ist kritisch**
- Nicht universell besser
- Abhängig von Environment, Buffer Size, Hyperparameter

✗ **Uniform Replay oft praktischer**
- Robuster
- Einfacher zu implementieren
- Schneller zu trainieren
- Weniger Hyperparameter zu tunen

### Best Practice:

1. **Start mit Uniform Replay**
   - Get a working baseline
   - Schnell & zuverlässig

2. **Wenn nicht zufrieden:** Evaluiere PER
   - Nur wenn clear benefit sichtbar
   - Sorgfältiges Hyperparameter-Tuning

3. **Monitor kontinuierlich**
   - Tracking: Rewards, Loss, Gradient Norms
   - Instabilität = Rückkehr zu Uniform

### Zukunftsrichtungen:

- **Hybrid Approaches**: Beginne mit Uniform, switch zu PER
- **Adaptive Alpha**: Lerne α während Training
- **Alternative Priorities**: Curiosity, KL-Divergence, etc.
- **Rainbow DQN**: Kombiniert PER mit Double Q, Dueling, etc.